# EvidencIA — Análise Exploratória de Dados (EDA) e Pipeline de Recuperação de Evidências

**Fase:** Fase 3 — Corpus, Retrieval e Avaliação  
**Data:** 2026-10-07  
**Reprodutibilidade:** Seed determinística (`SEED = 42`)  
**Status Metodológico:** Análise descritiva + Pipeline de Retrieval (Métricas de desempenho marcadas como `PENDING HUMAN ANNOTATION`)

---

In [1]:
import json
import random
from collections import Counter
from pathlib import Path

# Fixação de semente para reproducibilidade determinística
SEED = 42
random.seed(SEED)

print(f"[SETUP] Ambiente configurado com SEED={SEED}")

## 1. Carregamento e Inspeção do Corpus de Fatos (Fact-Checking Brasileiro)

Carregamos a base curada de fact-checking jornalístico (`sample_facts.json`) baseada no FactChecks.br e agências certificadas pelo IFCN.

In [2]:
facts_path = Path("../datasets/sample_facts.json")
if not facts_path.exists():
    # Fallback caso executado a partir da raiz do backend
    facts_path = Path("ml/datasets/sample_facts.json")

with open(facts_path, "r", encoding="utf-8") as f:
    facts = json.load(f)

print(f"Total de checagens catalogadas: {len(facts)}")
print(f"Exemplo de registro:\n{json.dumps(facts[0], indent=2, ensure_ascii=False)}")

## 2. Estatísticas Léxicas e Distribuições Categóricas

Examinamos a distribuição por categorias temáticas, agências checadoras (publishers) e tamanho de texto.

In [3]:
categories = [item.get("category", "indefinida") for item in facts]
publishers = [item.get("publisher", "desconhecido") for item in facts]
statuses = [item.get("status", "desconhecido") for item in facts]

claim_lengths = [len(item["claim"].split()) for item in facts]
evidence_lengths = [len(item.get("evidence_summary", "").split()) for item in facts]

print("=== Distribuição por Categoria ===")
for cat, count in Counter(categories).most_common():
    print(f"  - {cat}: {count} ({count/len(facts):.1%})")

print("\n=== Distribuição por Agência (Publisher) ===")
for pub, count in Counter(publishers).most_common():
    print(f"  - {pub}: {count} ({count/len(facts):.1%})")

print("\n=== Distribuição por Postura/Status Declarado ===")
for st, count in Counter(statuses).most_common():
    print(f"  - {st}: {count} ({count/len(facts):.1%})")

print(f"\nComprimento médio da alegação: {sum(claim_lengths)/len(claim_lengths):.1f} palavras")
print(f"Comprimento médio do resumo de evidência: {sum(evidence_lengths)/len(evidence_lengths):.1f} palavras")

## 3. Demonstração do Algoritmo de Recuperação Semântica e Lexical

Testamos a recuperação de candidatos para alegações representativas usando a correspondência léxica de Jaccard e tokens normalizados.

In [4]:
def tokenize(text: str) -> set[str]:
    stop_words = {"de", "a", "o", "que", "e", "do", "da", "em", "um", "para", "é", "com", "não", "uma", "os", "no", "se", "na", "por", "mais", "as", "dos", "como", "mas", "foi", "ao", "ele", "das", "tem", "à", "seu", "sua", "ou", "ser", "quando", "muito", "nos", "já", "está", "eu", "também", "só", "pelo", "pela", "até", "isso", "ela", "entre", "era", "depois", "sem", "mesmo", "aos", "ter", "seus", "quem", "nas", "me", "esse", "eles", "estão", "você", "tinha", "foram", "essa", "num", "nem", "suas", "meu", "às", "minha", "têm", "numa", "pelos", "elas", "havia", "seja", "qual", "será", "nós", "tenho", "lhe", "deles", "essas", "esses", "pelas", "este", "fosse", "dele", "tu", "te", "vocês", "vos", "lhes", "meus", "minhas", "teu", "tua", "teus", "tuas", "nosso", "nossa", "nossos", "nossas", "dela", "delas", "esta", "estes", "estas", "aquele", "aquela", "aqueles", "aquelas", "isto", "aquilo", "estou", "está", "estamos", "estão", "estive", "esteve", "estivemos", "estiveram", "estava", "estávamos", "estavam", "estivera", "estivéramos", "esteja", "estejamos", "estejam", "estivesse", "estivéssemos", "estivessem", "estiver", "estivermos", "estiverem", "hei", "há", "havemos", "hão", "houve", "houvemos", "houveram", "houvera", "houvéramos", "haja", "hajamos", "hajam", "houvesse", "houvéssemos", "houvessem", "houver", "houvermos", "houverem", "houverei", "houverá", "houveremos", "houverão", "houveria", "houveríamos", "houveriam", "sou", "somos", "são", "era", "éramos", "eram", "fui", "foi", "fomos", "foram", "fora", "fôramos", "seja", "sejamos", "sejam", "fosse", "fôssemos", "fossem", "for", "formos", "forem", "serei", "será", "seremos", "serão", "seria", "seríamos", "seriam", "tenho", "tem", "temos", "tém", "tinha", "tínhamos", "tinham", "tive", "teve", "tivemos", "tiveram", "tivera", "tivéramos", "tenha", "tenhamos", "tenham", "tivesse", "tivéssemos", "tivessem", "tiver", "tivermos", "tiverem", "terei", "terá", "teremos", "terão", "teria", "teríamos", "teriam"}
    words = [w.strip(".,!?:;\"'()[]{}").lower() for w in text.split()]
    return {w for w in words if len(w) > 2 and w not in stop_words}

def retrieve_top_k(query: str, facts: list[dict], k: int = 3) -> list[tuple[float, dict]]:
    q_tokens = tokenize(query)
    results = []
    for fact in facts:
        f_tokens = tokenize(fact["claim"] + " " + fact.get("evidence_summary", ""))
        inter = q_tokens.intersection(f_tokens)
        union = q_tokens.union(f_tokens)
        jaccard = len(inter) / max(len(union), 1)
        if jaccard > 0:
            results.append((jaccard, fact))
    results.sort(key=lambda x: x[0], reverse=True)
    return results[:k]

test_claim = "Água com limão pela manhã neutraliza a acidez e previne infecções"
print(f"Query de teste: '{test_claim}'")
retrieved = retrieve_top_k(test_claim, facts, k=3)
for rank, (score, doc) in enumerate(retrieved, start=1):
    print(f"  Rank {rank} (score={score:.3f}): [{doc['publisher']}] {doc['claim']}")

## 4. Estado da Avaliação de Recuperação e Stance (Gold Set)

Verificação dos arquivos de avaliação em `evaluation/`.

In [5]:
eval_dir = Path("../../evaluation")
if not eval_dir.exists():
    eval_dir = Path("evaluation")

candidates_file = eval_dir / "candidates.jsonl"
if candidates_file.exists():
    with open(candidates_file, "r", encoding="utf-8") as f:
        candidates = [json.loads(line) for line in f if line.strip()]
    
    annotated_count = sum(1 for c in candidates if c.get("human_relevance") is not None)
    print(f"Total de pares candidatos carregados: {len(candidates)}")
    print(f"Pares com anotação humana consolidada: {annotated_count}")
    
    if annotated_count == 0:
        print("\n[INTEGRIDADE CIENTÍFICA] Status: PENDING_HUMAN_ANNOTATION")
        print("Métricas de Recall@k, MRR e Macro-F1 não são calculadas para evitar relatórios forjados.")
        print("Consulte 'evaluation/annotation-guide.md' para orientações de anotação humana independente.")
else:
    print("Arquivo 'candidates.jsonl' não encontrado no diretório de avaliação.")